# OSU Avatar Feature Construction

This notebook contains executable Python code for every feature-construction
step. No Claude or external LLM generated row-level feature values. Complex
features are deterministic functions of standardized filename attributes and
objective image metrics. The output is a standalone feature file: only
`Version.ID` and `User.ID` overlap with the professor's main data file.


In [ ]:

from __future__ import annotations

import csv
import math
import sys
import re
import shutil
import zipfile
from pathlib import Path

import cv2
import numpy as np

BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR / "inputs"
if not INPUT_DIR.exists():
    INPUT_DIR = BASE_DIR
MAPPING_PATH = INPUT_DIR / "OSU_avatar_mapping_input.csv"
IMAGE_METRICS_PATH = INPUT_DIR / "OSU_avatar_image_metrics_input.csv"
OUTPUT_PATH = BASE_DIR / "OSU_avatar_features_final.csv"

# Set this to True to recompute primitive image metrics from the original
# Archive.zip and Archive copy.zip image archives.
RECOMPUTE_RAW_IMAGE_METRICS = False
IMAGE_ARCHIVES = [
    BASE_DIR / "Archive.zip",
    BASE_DIR / "Archive copy.zip",
]

def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        return list(csv.DictReader(handle))

def write_csv(path: Path, rows: list[dict], headers: list[str] | None = None) -> None:
    if headers is None:
        headers = list(rows[0].keys()) if rows else []
    with path.open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.DictWriter(handle, fieldnames=headers)
        writer.writeheader()
        writer.writerows(rows)

def as_number(value):
    if value is None:
        return None
    text = str(value).strip()
    if text in {"", "NA", "N/A", "None", "nan"}:
        return None
    try:
        number = float(text)
        return number if math.isfinite(number) else None
    except Exception:
        return None

def percentile_ranks(values: list[float]) -> np.ndarray:
    array = np.asarray(values, dtype=float)
    order = np.argsort(array)
    ranks = np.empty(len(array), dtype=float)
    index = 0
    while index < len(array):
        end = index
        while end + 1 < len(array) and array[order[end + 1]] == array[order[index]]:
            end += 1
        ranks[order[index:end + 1]] = (
            ((index + end) / 2) / max(len(array) - 1, 1) * 100
        )
        index = end + 1
    return ranks

def three_level(score, low, high, labels=("low", "medium", "high")):
    if score is None:
        return None
    if score < low:
        return labels[0]
    if score < high:
        return labels[1]
    return labels[2]

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"OpenCV: {cv2.__version__}")


In [ ]:

# ---------------------------------------------------------------------------
# Optional raw-image metric extraction
# ---------------------------------------------------------------------------
AVATAR_PATTERN = re.compile(
    r"^avatar_(?P<type>[^_]+)_(?P<gender>[^_]+)_(?P<emotion>[^_]+)_"
    r"(?P<color>[^_]+)_(?P<attire>[^_]+)_(?P<logo>[^_]+)_"
    r"(?P<variant>v\d+)\.jpeg$",
    re.IGNORECASE,
)

def extract_primitive_metrics(image_paths: list[Path]) -> list[dict]:
    """Create the primitive image metrics used by all downstream features."""
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )
    eye_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_eye.xml"
    )
    smile_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_smile.xml"
    )

    output = []
    for path in sorted(image_paths):
        match = AVATAR_PATTERN.match(path.name)
        if match is None:
            continue
        attributes = {
            key: value.lower()
            for key, value in match.groupdict().items()
        }

        bgr = cv2.imread(str(path))
        if bgr is None:
            raise ValueError(f"Could not read image: {path}")
        bgr = cv2.resize(bgr, (256, 256), interpolation=cv2.INTER_AREA)
        gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
        ycrcb = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb)

        y_channel, cr, cb = cv2.split(ycrcb)
        skin_mask = (
            (cr > 133) & (cr < 180)
            & (cb > 77) & (cb < 135)
            & (y_channel > 50)
        )

        grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        gradient = np.sqrt(grad_x ** 2 + grad_y ** 2)

        local_mean = cv2.blur(gray.astype(np.float32), (7, 7))
        local_square_mean = cv2.blur(gray.astype(np.float32) ** 2, (7, 7))
        local_sd = np.sqrt(
            np.maximum(local_square_mean - local_mean ** 2, 0)
        )

        hue_mask = (hsv[:, :, 1] > 30) & (hsv[:, :, 2] > 30)
        if hue_mask.any():
            hue_angle = (
                hsv[:, :, 0][hue_mask].astype(float)
                * 2 * np.pi / 180
            )
            weights = hsv[:, :, 1][hue_mask].astype(float) / 255
            circular_mean = (
                np.sum(weights * np.exp(1j * hue_angle))
                / np.sum(weights)
            )
            hue_concentration = float(abs(circular_mean))
        else:
            hue_concentration = 0.0

        crop = gray[:, 32:224]
        left = crop[:, :96]
        right = np.fliplr(crop[:, 96:192])
        symmetry = 1 - float(
            np.mean(np.abs(left.astype(float) - right.astype(float)))
            / 255
        )

        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=4,
            minSize=(40, 40),
        )
        face_detected = int(len(faces) > 0)
        eye_count = 0
        smile_detected = 0
        if len(faces):
            x, y, width, height = max(
                faces,
                key=lambda rectangle: rectangle[2] * rectangle[3],
            )
            face_roi = gray[y:y + height, x:x + width]
            eyes = eye_cascade.detectMultiScale(
                face_roi,
                1.1,
                4,
                minSize=(10, 10),
            )
            smiles = smile_cascade.detectMultiScale(
                face_roi,
                1.5,
                15,
                minSize=(20, 10),
            )
            eye_count = min(2, len(eyes))
            smile_detected = int(len(smiles) > 0)

        central_face_crop = gray[20:170, 45:211]

        output.append({
            "avatar_name": path.stem.removeprefix("avatar_"),
            "image_file_name": path.name,
            "avatar_type": attributes["type"],
            "avatar_gender": attributes["gender"],
            "avatar_emotion": attributes["emotion"],
            "clothing_color": attributes["color"],
            "attire_style": attributes["attire"],
            "logo_presence": (
                "Yes" if attributes["logo"] != "nologo" else "No"
            ),
            "logo_binary": (
                1 if attributes["logo"] != "nologo" else 0
            ),
            "image_variant": attributes["variant"],
            "mean_saturation_0_255": float(hsv[:, :, 1].mean()),
            "mean_brightness_0_255": float(hsv[:, :, 2].mean()),
            "hue_concentration_0_1": hue_concentration,
            "skin_tone_fraction_proxy": float(skin_mask.mean()),
            "symmetry_proxy_0_1": symmetry,
            "face_detected": face_detected,
            "eye_count_detected_0_2": eye_count,
            "smile_detected": smile_detected,
            "central_face_texture": float(
                cv2.Laplacian(
                    central_face_crop,
                    cv2.CV_64F,
                ).var()
            ),
            "laplacian_variance": float(
                cv2.Laplacian(gray, cv2.CV_64F).var()
            ),
            "flat_fraction": float((gradient < 8).mean()),
            "fine_texture_fraction": float((gradient > 40).mean()),
            "entropy_bits": float(
                (lambda probabilities:
                    -np.sum(
                        probabilities[probabilities > 0]
                        * np.log2(probabilities[probabilities > 0])
                    )
                )(
                    np.bincount(
                        gray.ravel(),
                        minlength=256,
                    ) / gray.size
                )
            ),
            "contrast_sd": float(gray.std()),
            "edge_density": float(
                (cv2.Canny(gray, 80, 160) > 0).mean()
            ),
            "local_texture": float(local_sd.mean()),
        })
    return output

if RECOMPUTE_RAW_IMAGE_METRICS:
    extraction_dir = BASE_DIR / "_avatar_images"
    if extraction_dir.exists():
        shutil.rmtree(extraction_dir)
    extraction_dir.mkdir()

    image_paths = []
    for archive_path in IMAGE_ARCHIVES:
        if not archive_path.exists():
            raise FileNotFoundError(
                f"Missing raw-image archive: {archive_path}"
            )
        target = extraction_dir / archive_path.stem
        target.mkdir()
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(target)
        image_paths.extend(target.rglob("*.jpeg"))

    primitive_rows = extract_primitive_metrics(image_paths)
    write_csv(
        IMAGE_METRICS_PATH,
        primitive_rows,
        list(primitive_rows[0].keys()),
    )
else:
    primitive_rows = read_csv(IMAGE_METRICS_PATH)

print(f"Primitive avatar rows loaded: {len(primitive_rows)}")


In [ ]:

# ---------------------------------------------------------------------------
# Image-level feature construction
# ---------------------------------------------------------------------------
numeric_primitive_fields = [
    "logo_binary",
    "mean_saturation_0_255",
    "mean_brightness_0_255",
    "hue_concentration_0_1",
    "skin_tone_fraction_proxy",
    "symmetry_proxy_0_1",
    "face_detected",
    "eye_count_detected_0_2",
    "smile_detected",
    "central_face_texture",
    "laplacian_variance",
    "flat_fraction",
    "fine_texture_fraction",
    "entropy_bits",
    "contrast_sd",
    "edge_density",
    "local_texture",
]

avatar_rows = []
for source in primitive_rows:
    row = dict(source)
    for field in numeric_primitive_fields:
        row[field] = as_number(row.get(field))
    avatar_rows.append(row)

rank_metrics = [
    "mean_saturation_0_255",
    "mean_brightness_0_255",
    "hue_concentration_0_1",
    "skin_tone_fraction_proxy",
    "symmetry_proxy_0_1",
    "central_face_texture",
    "laplacian_variance",
    "flat_fraction",
    "fine_texture_fraction",
    "entropy_bits",
    "contrast_sd",
    "edge_density",
    "local_texture",
]

for metric in rank_metrics:
    ranks = percentile_ranks([
        float(row[metric])
        for row in avatar_rows
    ])
    for index, row in enumerate(avatar_rows):
        row[f"{metric}_percentile"] = float(ranks[index])

for row in avatar_rows:
    realism = (
        row["local_texture_percentile"]
        + row["entropy_bits_percentile"]
        + row["fine_texture_fraction_percentile"]
        + (100 - row["flat_fraction_percentile"])
        + (100 - row["mean_saturation_0_255_percentile"])
    ) / 5

    craftsmanship_complexity = (
        row["laplacian_variance_percentile"]
        + row["edge_density_percentile"]
        + row["contrast_sd_percentile"]
        + row["entropy_bits_percentile"]
        + row["local_texture_percentile"]
    ) / 5

    human_likeness = (
        0.45 * row["skin_tone_fraction_proxy_percentile"]
        + 0.35 * realism
        + 0.20 * (100 if row["face_detected"] else 0)
    )
    mechanical_complexity = (
        row["edge_density_percentile"]
        + row["fine_texture_fraction_percentile"]
        + (100 - row["skin_tone_fraction_proxy_percentile"])
        + craftsmanship_complexity
    ) / 4

    row["realism_proxy_0_100"] = float(realism)
    row["craftsmanship_complexity_proxy_0_100"] = float(
        craftsmanship_complexity
    )
    row["human_likeness_proxy_0_100"] = float(human_likeness)
    row["roboticness_proxy_0_100"] = float(100 - human_likeness)
    row["mechanical_complexity_proxy_0_100"] = float(
        mechanical_complexity
    )

saturation_median = float(np.median([
    row["mean_saturation_0_255"]
    for row in avatar_rows
]))
brightness_median = float(np.median([
    row["mean_brightness_0_255"]
    for row in avatar_rows
]))
age_texture_median = float(np.median([
    row["central_face_texture"]
    for row in avatar_rows
]))
coordination_quintiles = np.quantile(
    [
        row["hue_concentration_0_1"]
        for row in avatar_rows
    ],
    [0.20, 0.40, 0.60, 0.80],
)

style_scores = []
for row in avatar_rows:
    color = row["clothing_color"]
    realism = row["realism_proxy_0_100"]
    craftsmanship = row["craftsmanship_complexity_proxy_0_100"]
    human_likeness = row["human_likeness_proxy_0_100"]
    mechanical = row["mechanical_complexity_proxy_0_100"]

    row["saturation_level"] = (
        "high"
        if row["mean_saturation_0_255"] >= saturation_median
        else "low"
    )
    row["brightness_level"] = (
        "high"
        if row["mean_brightness_0_255"] >= brightness_median
        else "low"
    )

    row["color_arousal_level"] = (
        "high"
        if color in {"red", "orange", "yellow"}
        else "moderate"
        if color in {"green", "purple"}
        else "low"
    )
    row["color_arousal_binary_high"] = int(
        color in {"red", "orange", "yellow"}
    )
    row["color_valence_level"] = (
        "neutral"
        if color in {"black", "white"}
        else "positive"
    )
    row["color_valence_binary_positive"] = int(
        color not in {"black", "white"}
    )
    row["color_temperature"] = (
        "warm"
        if color in {"red", "orange", "yellow"}
        else "cool"
        if color in {"blue", "green", "purple"}
        else "neutral"
    )
    row["warm_color_binary"] = int(
        row["color_temperature"] == "warm"
    )
    row["neutral_color_binary"] = int(
        row["color_temperature"] == "neutral"
    )

    row["color_coordination_score_1_5"] = 1 + sum(
        row["hue_concentration_0_1"] >= threshold
        for threshold in coordination_quintiles
    )

    row["human_likeness_level"] = three_level(
        human_likeness,
        33.33,
        66.67,
    )
    row["roboticness_level"] = (
        "high"
        if row["human_likeness_level"] == "low"
        else "medium"
        if row["human_likeness_level"] == "medium"
        else "low"
    )

    row["perceived_age_proxy"] = (
        "older-looking"
        if row["central_face_texture"] >= age_texture_median
        else "younger-looking"
    )
    row["perceived_age_older_binary"] = int(
        row["perceived_age_proxy"] == "older-looking"
    )
    row["perceived_ethnicity"] = (
        "ambiguous/not reliably inferable"
    )

    row["dimension_style_proxy"] = (
        "2D/flat"
        if (
            row["flat_fraction_percentile"] >= 60
            and row["local_texture_percentile"] <= 40
        )
        else "3D/depth-rendered"
    )
    row["dimension_3d_binary"] = int(
        row["dimension_style_proxy"] == "3D/depth-rendered"
    )

    row["facial_feature_count_proxy_0_4"] = (
        int(row["face_detected"])
        + int(row["eye_count_detected_0_2"] > 0)
        + int(row["smile_detected"])
        + int(row["central_face_texture_percentile"] >= 50)
    )
    row["visible_mechanical_complexity_0_3"] = (
        0
        if mechanical < 25
        else 1
        if mechanical < 50
        else 2
        if mechanical < 75
        else 3
    )
    row["attire_completeness_0_2"] = (
        int(row["attire_style"] == "formal")
        + int(row["logo_binary"] == 1)
    )
    row["tie_bow_tie_proxy"] = int(
        row["attire_style"] == "formal"
    )

    row["role_congruence_score_1_5"] = min(
        5,
        1
        + int(row["attire_style"] == "formal")
        + int(row["logo_binary"] == 1)
        + int(row["avatar_emotion"] == "happy")
        + int(row["dimension_3d_binary"] == 1),
    )
    row["warmth_approachability_score_1_7"] = min(
        7,
        1
        + 2 * int(row["avatar_emotion"] == "happy")
        + int(row["warm_color_binary"] == 1)
        + int(row["brightness_level"] == "high")
        + 2 * human_likeness / 100,
    )

    aesthetic_percent = (
        0.35
        * ((row["color_coordination_score_1_5"] - 1) / 4 * 100)
        + 0.30 * row["symmetry_proxy_0_1_percentile"]
        + 0.35 * craftsmanship
    )
    row["aesthetic_coherence_score_1_7"] = (
        1 + 6 * aesthetic_percent / 100
    )

    style_score = (
        0.40 * realism
        + 0.30 * craftsmanship
        + 0.30 * human_likeness
    )
    row["rendering_style_score"] = float(style_score)
    style_scores.append(style_score)

style_quartiles = np.quantile(
    style_scores,
    [0.25, 0.50, 0.75],
)
style_labels = [
    "flat 2D illustration",
    "3D cartoon/stylized",
    "3D realistic robot",
    "3D photorealistic android",
]

for row in avatar_rows:
    score = row["rendering_style_score"]
    tier = (
        0
        if score < style_quartiles[0]
        else 1
        if score < style_quartiles[1]
        else 2
        if score < style_quartiles[2]
        else 3
    )
    row["rendering_style_tier_0_3"] = tier
    row["rendering_style_label"] = style_labels[tier]
    row["method_note"] = (
        "Image-derived proxy. Ethnicity was not inferred; "
        "age/style labels are exploratory."
    )

    realism = row["realism_proxy_0_100"]
    roboticness = row["roboticness_proxy_0_100"]
    mechanical = row["mechanical_complexity_proxy_0_100"]

    row["realism_style"] = three_level(
        realism,
        33.333,
        66.667,
        ("cartoonish", "semi-realistic", "realistic"),
    )
    row["cartoonish_binary"] = int(
        row["realism_style"] == "cartoonish"
    )
    row["realistic_binary"] = int(
        row["realism_style"] == "realistic"
    )

    craftsmanship_weighted = 100 * (
        0.50 * (tier / 3)
        + 0.20 * (
            row["facial_feature_count_proxy_0_4"] / 4
        )
        + 0.20 * (
            row["visible_mechanical_complexity_0_3"] / 3
        )
        + 0.10 * (
            row["attire_completeness_0_2"] / 2
        )
    )
    row["craftsmanship_weighted_0_100"] = round(
        craftsmanship_weighted,
        3,
    )
    row["craftsmanship_quality_level"] = three_level(
        craftsmanship_weighted,
        33.333,
        66.667,
    )

    row["eyelashes_present_visual_proxy"] = int(
        row["avatar_gender"] == "female"
        and (
            tier >= 1
            or realism >= 35
        )
    )
    row["visible_iris_present_visual_proxy"] = int(
        row["eye_count_detected_0_2"] > 0
        or tier >= 2
        or realism >= 55
    )
    row["skin_shading_present_visual_proxy"] = int(
        row["dimension_3d_binary"] == 1
        or tier >= 1
    )
    row["blush_present_visual_proxy"] = int(
        row["avatar_gender"] == "female"
        and row["avatar_emotion"] == "happy"
        and row["mean_saturation_0_255"] >= 55
        and row["mean_brightness_0_255"] >= 175
    )
    row["eyebrows_present_visual_proxy"] = int(
        row["face_detected"] == 1
        or tier >= 1
        or realism >= 40
    )
    row["visible_teeth_present_visual_proxy"] = int(
        row["avatar_emotion"] == "happy"
        and (
            row["smile_detected"] == 1
            or tier >= 3
        )
    )
    row["mechanical_neck_present_visual_proxy"] = int(
        roboticness >= 45
        and (
            row["dimension_3d_binary"] == 1
            or tier >= 2
        )
    )
    row["panel_seams_present_visual_proxy"] = int(
        row["visible_mechanical_complexity_0_3"] >= 2
        or mechanical >= 55
    )
    row["headset_present_template_code"] = 1
    row["name_badge_present_visual_proxy"] = int(
        row["logo_binary"] == 1
        or row["attire_style"] == "formal"
        or tier >= 2
    )
    row["tie_bow_tie_present_filename_rule"] = int(
        row["attire_style"] == "formal"
    )
    row["logo_present_filename_rule"] = int(
        row["logo_binary"]
    )

    inventory_fields = [
        "eyelashes_present_visual_proxy",
        "visible_iris_present_visual_proxy",
        "skin_shading_present_visual_proxy",
        "blush_present_visual_proxy",
        "eyebrows_present_visual_proxy",
        "visible_teeth_present_visual_proxy",
        "mechanical_neck_present_visual_proxy",
        "panel_seams_present_visual_proxy",
        "headset_present_template_code",
        "name_badge_present_visual_proxy",
        "tie_bow_tie_present_filename_rule",
        "logo_present_filename_rule",
    ]
    row["feature_inventory_count_0_12"] = sum(
        int(row[field])
        for field in inventory_fields
    )

    row["apparent_service_role"] = (
        "restaurant/service staff"
    )
    role_signals = (
        int(row["attire_style"] == "formal")
        + int(row["logo_binary"] == 1)
        + int(row["name_badge_present_visual_proxy"] == 1)
        + int(row["headset_present_template_code"] == 1)
    )
    row["apparent_role_confidence"] = (
        "high"
        if role_signals >= 3
        else "medium"
        if role_signals >= 2
        else "low"
    )

    coordination = row["color_coordination_score_1_5"]
    row["color_coordination_explanation"] = (
        "Highly concentrated palette with visually consistent "
        "clothing and accents."
        if coordination >= 4
        else "Moderately coordinated palette with some competing "
        "accent colors."
        if coordination >= 3
        else "Low palette concentration; several visual colors "
        "compete for attention."
    )

    aesthetic = row["aesthetic_coherence_score_1_7"]
    row["aesthetic_coherence_explanation"] = (
        "Facial style, attire, color, and accessories form a "
        "strongly unified design."
        if aesthetic >= 5
        else "The major design elements generally work together, "
        "with some stylistic variation."
        if aesthetic >= 3.5
        else "The design elements appear less unified or use "
        "mixed visual styles."
    )

    role_score = row["role_congruence_score_1_5"]
    row["role_congruence_explanation"] = (
        "Uniform, headset, badge/logo, and formal styling strongly "
        "signal a service role."
        if role_score >= 4
        else "Some service-role signals are visible, but the role "
        "presentation is less explicit."
    )

    warmth = row["warmth_approachability_score_1_7"]
    row["warmth_approachability_explanation"] = (
        "Happy expression, brighter styling, and human-like facial "
        "cues make the avatar approachable."
        if warmth >= 5
        else "The avatar appears moderately approachable, with a "
        "mix of friendly and mechanical cues."
        if warmth >= 3.5
        else "Neutral expression or stronger mechanical cues reduce "
        "perceived approachability."
    )

    row["perceived_age_group"] = row["perceived_age_proxy"]
    row["perceived_ethnicity_coding"] = (
        "ambiguous/not reliably inferable"
    )
    row["ethnicity_coding_confidence"] = (
        "insufficient visual basis"
    )
    row["ethnic_match_applicable"] = (
        "No — avatar ethnicity not reliably inferable"
    )
    row["visual_coding_method"] = (
        "Filename attributes + objective image metrics + explicit "
        "reproducible visual-proxy rules"
    )
    row["visual_coding_limitation"] = (
        "Exploratory generated-avatar coding; not a validated "
        "human-rating scale."
    )

avatar_lookup = {
    row["avatar_name"]: row
    for row in avatar_rows
}
print(f"Constructed image-level features for {len(avatar_rows)} avatars.")


In [ ]:

# ---------------------------------------------------------------------------
# Participant-level feature construction
# ---------------------------------------------------------------------------
mapping_rows = read_csv(MAPPING_PATH)

version_id_to_condition = {
    "v1r": "create_av",
    "v2r": "human_av",
    "v4r": "no_av",
}
gender_code_to_label = {
    "a": "female",
    "b": "male",
    "c": "nonbinary / third gender",
}

image_feature_fields = [
    "avatar_gender",
    "avatar_emotion",
    "clothing_color",
    "attire_style",
    "logo_presence",
    "logo_binary",
    "image_variant",
    "mean_saturation_0_255",
    "mean_brightness_0_255",
    "saturation_level",
    "brightness_level",
    "color_arousal_level",
    "color_arousal_binary_high",
    "color_valence_level",
    "color_valence_binary_positive",
    "color_temperature",
    "warm_color_binary",
    "neutral_color_binary",
    "realism_proxy_0_100",
    "craftsmanship_complexity_proxy_0_100",
    "human_likeness_proxy_0_100",
    "roboticness_proxy_0_100",
    "human_likeness_level",
    "roboticness_level",
    "dimension_style_proxy",
    "dimension_3d_binary",
    "rendering_style_tier_0_3",
    "rendering_style_label",
    "perceived_age_proxy",
    "perceived_age_older_binary",
    "perceived_ethnicity",
    "facial_feature_count_proxy_0_4",
    "visible_mechanical_complexity_0_3",
    "attire_completeness_0_2",
    "tie_bow_tie_proxy",
    "color_coordination_score_1_5",
    "aesthetic_coherence_score_1_7",
    "role_congruence_score_1_5",
    "warmth_approachability_score_1_7",
    "realism_style",
    "cartoonish_binary",
    "realistic_binary",
    "craftsmanship_weighted_0_100",
    "craftsmanship_quality_level",
    "eyelashes_present_visual_proxy",
    "visible_iris_present_visual_proxy",
    "skin_shading_present_visual_proxy",
    "blush_present_visual_proxy",
    "eyebrows_present_visual_proxy",
    "visible_teeth_present_visual_proxy",
    "mechanical_neck_present_visual_proxy",
    "panel_seams_present_visual_proxy",
    "headset_present_template_code",
    "name_badge_present_visual_proxy",
    "tie_bow_tie_present_filename_rule",
    "logo_present_filename_rule",
    "feature_inventory_count_0_12",
    "apparent_service_role",
    "apparent_role_confidence",
    "color_coordination_explanation",
    "aesthetic_coherence_explanation",
    "role_congruence_explanation",
    "warmth_approachability_explanation",
    "perceived_age_group",
    "perceived_ethnicity_coding",
    "ethnicity_coding_confidence",
    "ethnic_match_applicable",
    "visual_coding_method",
    "visual_coding_limitation",
]

def valid_avatar_name(value):
    text = str(value or "").strip()
    if text in {
        "",
        "NA",
        "No avatar",
        "Default avatar",
    }:
        return None
    return text


participant_features = []
for mapping in mapping_rows:
    version_id = mapping["Version.ID"]
    user_id = mapping["User.ID"]
    condition = version_id_to_condition[version_id]

    avatar_names = [
        valid_avatar_name(mapping.get("avatar_1")),
        valid_avatar_name(mapping.get("avatar_2")),
        valid_avatar_name(mapping.get("avatar_3")),
    ]
    first_name = avatar_names[0]
    final_name = (
        valid_avatar_name(mapping.get("final_avatar_name_source"))
        if condition == "create_av"
        else None
    )

    first_record = (
        avatar_lookup.get(first_name)
        if condition in {"create_av", "human_av"} and first_name is not None
        else None
    )
    final_record = (
        avatar_lookup.get(final_name)
        if condition == "create_av" and final_name is not None
        else None
    )

    user_gender = gender_code_to_label.get(
        str(mapping.get("user_gender_code") or "").strip()
    )

    output = {
        "Version.ID": version_id,
        "User.ID": user_id,
        "mapping_status": mapping.get("mapping_status_source") or None,
        "mapping_confidence": mapping.get("mapping_confidence_source") or None,
        "structure_match": mapping.get("structure_match_source") or None,
        "match_diff_minutes": mapping.get("match_diff_minutes_source") or None,
        "robot_session_id": mapping.get("robot_session_id_source") or None,
        "session_file_name": mapping.get("session_file_name_source") or None,
        "avatar_1": mapping.get("avatar_1") or None,
        "avatar_2": mapping.get("avatar_2") or None,
        "avatar_3": mapping.get("avatar_3") or None,
        "distinct_avatars_created_v1": mapping.get(
            "distinct_avatars_created_v1_source"
        ) or None,
        "interaction_intensity_v1": mapping.get(
            "interaction_intensity_v1_source"
        ) or None,
        "user_gender_label": user_gender,
        "first_avatar_gender_match": None,
        "any_avatar_gender_match": None,
        "final_avatar_gender_match": None,
        "previous_qc_inclusion": mapping.get(
            "previous_qc_inclusion_source"
        ) or None,
        "previous_inner_join_missing_id": mapping.get(
            "previous_inner_join_missing_id_source"
        ) or None,
        "revised_feature_table_row_present": mapping.get(
            "revised_feature_table_row_present_source"
        ) or None,
        "first_avatar_name": first_name if first_record is not None else None,
        "final_avatar_name": final_name if final_record is not None else None,
    }

    first_gender = first_record["avatar_gender"] if first_record is not None else None
    final_gender = final_record["avatar_gender"] if final_record is not None else None

    output["first_avatar_gender_match"] = (
        int(user_gender == first_gender)
        if user_gender is not None and first_gender is not None
        else None
    )

    # Keep the previous any-avatar feature. For human_av there is one avatar,
    # so "any" equals the first-avatar match; for create_av it checks all
    # identifiable created avatars.
    if user_gender is not None and condition in {"create_av", "human_av"}:
        created_genders = [
            avatar_lookup[name]["avatar_gender"]
            for name in avatar_names
            if name in avatar_lookup
        ]
        output["any_avatar_gender_match"] = (
            int(user_gender in created_genders) if created_genders else None
        )

    output["final_avatar_gender_match"] = (
        int(user_gender == final_gender)
        if condition == "create_av"
        and user_gender is not None
        and final_gender is not None
        else None
    )

    for prefix_name, record in [("first", first_record), ("final", final_record)]:
        for field in image_feature_fields:
            output[f"{prefix_name}_{field}"] = (
                record[field] if record is not None else None
            )

    for prefix_name in ["first", "final"]:
        emotion = output.get(f"{prefix_name}_avatar_emotion")
        attire = output.get(f"{prefix_name}_attire_style")
        saturation = output.get(f"{prefix_name}_saturation_level")
        brightness = output.get(f"{prefix_name}_brightness_level")

        output[f"{prefix_name}_avatar_happy_binary"] = (
            1 if emotion == "happy" else 0 if emotion == "neutral" else None
        )
        output[f"{prefix_name}_avatar_formal_binary"] = (
            1 if attire == "formal" else 0 if attire == "informal" else None
        )
        output[f"{prefix_name}_saturation_high_binary"] = (
            1 if saturation == "high" else 0 if saturation == "low" else None
        )
        output[f"{prefix_name}_brightness_high_binary"] = (
            1 if brightness == "high" else 0 if brightness == "low" else None
        )

    output["first_ethnic_match"] = None
    output["final_ethnic_match"] = None
    output["perceived_ethnic_match"] = None
    output["ethnicity_coding_note"] = (
        "Not coded: ethnicity is ambiguous and not reliably inferable "
        "from these robot avatars."
    )
    output["ethnic_match_note"] = (
        "Not calculated because avatar ethnicity is ambiguous and "
        "not reliably inferable."
    )

    if condition == "no_av":
        output["feature_availability_status"] = (
            "Not applicable - no avatar condition"
        )
    elif first_record is None:
        output["feature_availability_status"] = "Avatar image unavailable"
    elif mapping.get("structure_match_source") != "Yes":
        output["feature_availability_status"] = (
            "Available - mapping review flag retained"
        )
    else:
        output["feature_availability_status"] = "Available"

    participant_features.append(output)

# Add first-avatar aliases for legacy final-feature names already present in the
# main analysis file. This guarantees a pooled first_* counterpart for every
# create_av-only final_* feature name after the R merge.
for row in participant_features:
    row["first_avatar_attire"] = row.get("first_attire_style")
    row["first_avatar_color"] = row.get("first_clothing_color")
    row["first_avatar_logo_binary"] = row.get("first_logo_binary")
    row["first_avatar_variant"] = row.get("first_image_variant")

# Standardize selected continuous features separately for the pooled first set
# and the create_av-only final set.
standardize_suffixes = [
    "realism_proxy_0_100",
    "craftsmanship_complexity_proxy_0_100",
    "human_likeness_proxy_0_100",
    "color_coordination_score_1_5",
    "aesthetic_coherence_score_1_7",
    "role_congruence_score_1_5",
    "warmth_approachability_score_1_7",
    "craftsmanship_weighted_0_100",
    "feature_inventory_count_0_12",
]

for prefix_name in ["first", "final"]:
    for suffix in standardize_suffixes:
        field = f"{prefix_name}_{suffix}"
        values = [
            as_number(row.get(field))
            for row in participant_features
            if as_number(row.get(field)) is not None
        ]
        mean = float(np.mean(values))
        sd = float(np.std(values, ddof=1))
        for row in participant_features:
            value = as_number(row.get(field))
            row[f"{field}_z"] = (
                (value - mean) / sd
                if value is not None and sd > 0
                else None
            )

# ---------------------------------------------------------------------------
# Required quality checks
# ---------------------------------------------------------------------------
assert len(participant_features) == 190
assert len({
    (row["Version.ID"], row["User.ID"])
    for row in participant_features
}) == 190

for forbidden in ["version_enc", "wd_flag"]:
    assert forbidden not in participant_features[0]

first_suffixes = {
    column.removeprefix("first_")
    for column in participant_features[0]
    if column.startswith("first_")
}
final_suffixes = {
    column.removeprefix("final_")
    for column in participant_features[0]
    if column.startswith("final_")
}
assert final_suffixes.issubset(first_suffixes)

for row in participant_features:
    condition = version_id_to_condition[row["Version.ID"]]
    if condition != "create_av":
        for column, value in row.items():
            if column.startswith("final_"):
                assert value in {None, ""}

for row in participant_features:
    condition = version_id_to_condition[row["Version.ID"]]
    if (
        condition in {"create_av", "human_av"}
        and row["User.ID"] not in {"8", "148"}
    ):
        assert row["first_avatar_name"] not in {None, ""}
        assert row["first_logo_binary"] is not None
        assert row["first_realism_style"] is not None

# Exact schema of the professor's main file and the previous feature run.
# These constants make the standalone-file exclusion rule reproducible without
# requiring the main data during feature construction.
MAIN_DATA_COLUMNS = set(['Version.ID', 'User.ID', 'date', 'time1', 'Fitbit1', 'time2', 'Fitbit2', 'X1.1', 'X1.2', 'X1.3', 'X1.4', 'X1.5', 'X1.6', 'X1.7', 'X2.1', 'X2.2', 'X2.3', 'X2.4', 'X2.5', 'X2.6', 'X2.7', 'x_1_1_1_the_robot_avatar_felt_tailored_to_me', 'x1_1_2_the_robot_avatar_felt_personalized', 'x1_1_3_customized_robot_avatar_to_me', 'x1_2_1_robot_avatar_expresses_my_unique_personality', 'x_1_2_2_it_makes_me_feel_unique', 'x1_2_3_the_robot_avatar_reflects_my_uniqueness', 'x1_3_1_i_like_the_robot_avatar', 'x1_3_2_my_idea_of_a_perfect_robot_avatar', 'x1_3_3_the_robot_avatar_looks_really_great', 'x1_4_1_he_design_of_the_robot_avatar_is_appealing', 'x1_4_2_the_design_of_the_robot_avatar_is_attractive', 'x1_4_3_the_design_of_the_robot_avatar_looks_great', 'x1_5_1_i_feel_a_strong_sense_of_closeness_to_this_robot_avatar', 'x1_5_2_i_feel_a_high_degree_of_personal_ownership_over_this_robot_avatar', 'x1_5_3_this_robot_avatar_incorporates_a_part_of_myself', 'x1_6_1_you_co_created_the_robot_avatar', 'X2.2.1', 'X2.2.2', 'X2.2.3', 'X2.3.1', 'X2.3.2', 'X2.3.3', 'X2.4.1', 'X2.4.2', 'X2.4.3', 'X2.5.1', 'X2.5.2', 'X2.5.3', 'X2.6.1', 'X2.6.2', 'X2.6.3', 'X2.7.1', 'X2.7.2', 'X2.7.3', 'X2.8.1', 'X2.8.2', 'X2.8.3', 'X2.8.4', 'X2.8.5', 'X2.9', 'X2.9.1', 'X2.9.2', 'X2.10', 'tip_robot', 'tip_human', 'tip_discount', 'X2.11.a', 'X2.11.b', 'X2.11.c', 'X2.12.1', 'X2.12.2', 'X2.12.3', 'X2.13.1', 'X2.13.2', 'X2.13.3', 'X2.14.1', 'X2.14.2', 'X2.14.3', 'X2.15', 'X2.16.1', 'X2.16.2', 'X2.16.3', 'X2.17', 'X2.18', 'X2.19', 'X2.20', 'X2.21', 'X2.22.a', 'X2.22.b', 'overall_experience', 'robot_interface', 'ordering_process', 'general_take_on_the_interaction_being_served_by_robots_and_humans', 'drawbacks_of_being_served_by_robots', 'what_elements_of_the_creation_process_you_liked_and_disliked', 'multiple_versions_of_avatars', 'why_did_you_choose_a_specific_one', 'did_you_find_the_creation_options_like_gender_logo_helpful_or_was_something_missing_would_you_change_anything_about_the_options', 'when_you_created_the_avatar_did_it_make_you_feel_that_like_i_did_this_like_this_is_my_own_creation', 'wd_flag_raw', 'x', 'x_2', 'x_3', 'x_4', 'x_5', 'x_6', 'x_7', 'x_8', 'x_9', 'x_10', 'x_11', 'x_12', 'x_13', 'x_14', 'x_15', 'x_16', 'x_17', 'x_18', 'x_19', 'x_20', 'x_21', 'x_22', 'x_23', 'x_24', 'x_25', 'x_26', 'x_27', 'x_28', 'x_29', 'x_30', 'x_31', 'x_32', 'x_33', 'x_34', 'x_35', 'x_36', 'x_37', 'x_38', 'x_39', 'x_40', 'x_41', 'x_42', 'x_43', 'x_44', 'x_45', 'x_46', 'x_47', 'x_48', 'x_49', 'x_50', 'x_51', 'x_52', 'x_53', 'x_54', 'x_55', 'x_56', 'x_57', 'x_58', 'x_59', 'x_60', 'x_61', 'x_62', 'x_63', 'x_64', 'x_65', 'x_66', 'x_67', 'x_68', 'x_69', 'x_70', 'x_71', 'x_72', 'x_73', 'x_74', 'x_75', 'x_76', 'x_77', 'x_78', 'x_79', 'x_80', 'x_81', 'x_82', 'x_83', 'x_84', 'x_85', 'x_86', 'x_87', 'x_88', 'x_89', 'x_90', 'x_91', 'x_92', 'x_93', 'x_94', 'x_95', 'x_96', 'x_97', 'x_98', 'x_99', 'x_100', 'x_101', 'x_102', 'x_103', 'x_104', 'x_105', 'x_106', 'x_107', 'x_108', 'x_109', 'x_110', 'x_111', 'x_112', 'x_113', 'x_114', 'x_115', 'x_116', 'x_117', 'x_118', 'x_119', 'x_120', 'x_121', 'x_122', 'x_123', 'x_124', 'x_125', 'x_126', 'x_127', 'x_128', 'x_129', 'x_130', 'x_131', 'x_132', 'x_133', 'x_134', 'x_135', 'x_136', 'x_137', 'x_138', 'x_139', 'x_140', 'x_141', 'x_142', 'x_143', 'x_144', 'x_145', 'x_146', 'x_147', 'x_148', 'x_149', 'x_150', 'x_151', 'x_152', 'x_153', 'x_154', 'x_155', 'x_156', 'wd_flag', 'version_code', 'version_enc', 'X2.16.3_r', 'flawless', 'sat', 'sat2', 'serv_qual', 'reuse', 'unique_perc', 'custom_perc', 'close_perc', 'like_perc', 'control_perc', 'effort_perc', 'design_attract', 'subj_stress1', 'racewhite', 'gender', 'revisit', 'age', 'educ', 'robot_familiar', 'robot_comfort', 'time_of_day', 'mapping_status', 'mapping_confidence', 'structure_match', 'match_diff_minutes', 'robot_session_id', 'session_file_name', 'avatar_1', 'avatar_2', 'avatar_3', 'avatar_1_gender', 'avatar_2_gender', 'avatar_3_gender', 'distinct_avatars_created_v1', 'interaction_intensity_v1', 'user_gender_label', 'first_avatar_gender_match', 'any_avatar_gender_match', 'final_avatar_gender_match', 'final_avatar_name', 'final_avatar_gender', 'final_avatar_emotion', 'final_avatar_color', 'final_avatar_attire', 'final_avatar_logo_binary', 'final_avatar_variant', 'avatar_female_binary', 'avatar_happy_binary', 'avatar_formal_binary', 'realism_proxy_0_100', 'realism_style', 'craftsmanship_complexity_proxy_0_100', 'craftsmanship_level', 'realism_proxy_z', 'craftsmanship_proxy_z', 'total_time_seconds', 'time_AVATAR_CUSTOMIZER_seconds', 'page_sequence', 'avatar_feature_available', 'analysis_note'])
PREVIOUS_RUN_COLUMNS = ['Version.ID', 'User.ID', 'version_enc', 'wd_flag', 'mapping_status', 'mapping_confidence', 'structure_match', 'match_diff_minutes', 'robot_session_id', 'session_file_name', 'avatar_1', 'avatar_2', 'avatar_3', 'distinct_avatars_created_v1', 'interaction_intensity_v1', 'user_gender_label', 'first_avatar_gender_match', 'any_avatar_gender_match', 'final_avatar_gender_match', 'previous_qc_inclusion', 'previous_inner_join_missing_id', 'revised_feature_table_row_present', 'first_avatar_name', 'final_avatar_name', 'first_avatar_gender', 'first_avatar_emotion', 'first_clothing_color', 'first_attire_style', 'first_logo_presence', 'first_logo_binary', 'first_image_variant', 'first_mean_saturation_0_255', 'first_mean_brightness_0_255', 'first_saturation_level', 'first_brightness_level', 'first_color_arousal_level', 'first_color_arousal_binary_high', 'first_color_valence_level', 'first_color_valence_binary_positive', 'first_color_temperature', 'first_warm_color_binary', 'first_neutral_color_binary', 'first_realism_proxy_0_100', 'first_craftsmanship_complexity_proxy_0_100', 'first_human_likeness_proxy_0_100', 'first_roboticness_proxy_0_100', 'first_human_likeness_level', 'first_roboticness_level', 'first_dimension_style_proxy', 'first_dimension_3d_binary', 'first_rendering_style_tier_0_3', 'first_rendering_style_label', 'first_perceived_age_proxy', 'first_perceived_age_older_binary', 'first_perceived_ethnicity', 'first_facial_feature_count_proxy_0_4', 'first_visible_mechanical_complexity_0_3', 'first_attire_completeness_0_2', 'first_tie_bow_tie_proxy', 'first_color_coordination_score_1_5', 'first_aesthetic_coherence_score_1_7', 'first_role_congruence_score_1_5', 'first_warmth_approachability_score_1_7', 'final_avatar_gender', 'final_avatar_emotion', 'final_clothing_color', 'final_attire_style', 'final_logo_presence', 'final_logo_binary', 'final_image_variant', 'final_mean_saturation_0_255', 'final_mean_brightness_0_255', 'final_saturation_level', 'final_brightness_level', 'final_color_arousal_level', 'final_color_arousal_binary_high', 'final_color_valence_level', 'final_color_valence_binary_positive', 'final_color_temperature', 'final_warm_color_binary', 'final_neutral_color_binary', 'final_realism_proxy_0_100', 'final_craftsmanship_complexity_proxy_0_100', 'final_human_likeness_proxy_0_100', 'final_roboticness_proxy_0_100', 'final_human_likeness_level', 'final_roboticness_level', 'final_dimension_style_proxy', 'final_dimension_3d_binary', 'final_rendering_style_tier_0_3', 'final_rendering_style_label', 'final_perceived_age_proxy', 'final_perceived_age_older_binary', 'final_perceived_ethnicity', 'final_facial_feature_count_proxy_0_4', 'final_visible_mechanical_complexity_0_3', 'final_attire_completeness_0_2', 'final_tie_bow_tie_proxy', 'final_color_coordination_score_1_5', 'final_aesthetic_coherence_score_1_7', 'final_role_congruence_score_1_5', 'final_warmth_approachability_score_1_7', 'first_avatar_happy_binary', 'first_avatar_formal_binary', 'first_saturation_high_binary', 'first_brightness_high_binary', 'final_avatar_happy_binary', 'final_avatar_formal_binary', 'final_saturation_high_binary', 'final_brightness_high_binary', 'perceived_ethnic_match', 'ethnicity_coding_note', 'feature_availability_status', 'first_realism_proxy_0_100_z', 'first_craftsmanship_complexity_proxy_0_100_z', 'first_human_likeness_proxy_0_100_z', 'first_color_coordination_score_1_5_z', 'first_aesthetic_coherence_score_1_7_z', 'first_role_congruence_score_1_5_z', 'first_warmth_approachability_score_1_7_z', 'final_realism_proxy_0_100_z', 'final_craftsmanship_complexity_proxy_0_100_z', 'final_human_likeness_proxy_0_100_z', 'final_color_coordination_score_1_5_z', 'final_aesthetic_coherence_score_1_7_z', 'final_role_congruence_score_1_5_z', 'final_warmth_approachability_score_1_7_z', 'first_realism_style', 'first_cartoonish_binary', 'first_realistic_binary', 'first_craftsmanship_weighted_0_100', 'first_craftsmanship_quality_level', 'first_eyelashes_present_visual_proxy', 'first_visible_iris_present_visual_proxy', 'first_skin_shading_present_visual_proxy', 'first_blush_present_visual_proxy', 'first_eyebrows_present_visual_proxy', 'first_visible_teeth_present_visual_proxy', 'first_mechanical_neck_present_visual_proxy', 'first_panel_seams_present_visual_proxy', 'first_headset_present_template_code', 'first_name_badge_present_visual_proxy', 'first_tie_bow_tie_present_filename_rule', 'first_logo_present_filename_rule', 'first_feature_inventory_count_0_12', 'first_apparent_service_role', 'first_apparent_role_confidence', 'first_color_coordination_explanation', 'first_aesthetic_coherence_explanation', 'first_role_congruence_explanation', 'first_warmth_approachability_explanation', 'first_perceived_age_group', 'first_perceived_ethnicity_coding', 'first_ethnicity_coding_confidence', 'first_ethnic_match_applicable', 'first_visual_coding_method', 'first_visual_coding_limitation', 'final_realism_style', 'final_cartoonish_binary', 'final_realistic_binary', 'final_craftsmanship_weighted_0_100', 'final_craftsmanship_quality_level', 'final_eyelashes_present_visual_proxy', 'final_visible_iris_present_visual_proxy', 'final_skin_shading_present_visual_proxy', 'final_blush_present_visual_proxy', 'final_eyebrows_present_visual_proxy', 'final_visible_teeth_present_visual_proxy', 'final_mechanical_neck_present_visual_proxy', 'final_panel_seams_present_visual_proxy', 'final_headset_present_template_code', 'final_name_badge_present_visual_proxy', 'final_tie_bow_tie_present_filename_rule', 'final_logo_present_filename_rule', 'final_feature_inventory_count_0_12', 'final_apparent_service_role', 'final_apparent_role_confidence', 'final_color_coordination_explanation', 'final_aesthetic_coherence_explanation', 'final_role_congruence_explanation', 'final_warmth_approachability_explanation', 'final_perceived_age_group', 'final_perceived_ethnicity_coding', 'final_ethnicity_coding_confidence', 'final_ethnic_match_applicable', 'final_visual_coding_method', 'final_visual_coding_limitation', 'first_ethnic_match', 'final_ethnic_match', 'ethnic_match_note', 'first_craftsmanship_weighted_0_100_z', 'final_craftsmanship_weighted_0_100_z', 'first_feature_inventory_count_0_12_z', 'final_feature_inventory_count_0_12_z']
NEW_POOLED_ALIAS_COLUMNS = [
    "first_avatar_attire",
    "first_avatar_color",
    "first_avatar_logo_binary",
    "first_avatar_variant",
]

# Confirm every previous-run column is either already in the professor's main
# file or is recreated by this notebook.
constructed_columns = set(participant_features[0])
missing_previous_combined = [
    column
    for column in PREVIOUS_RUN_COLUMNS
    if column not in MAIN_DATA_COLUMNS and column not in constructed_columns
]
assert missing_previous_combined == []

# The standalone feature file contains the two merge IDs plus only columns not
# already present in d_osu_final.csv. This prevents .x/.y duplicate columns in R.
standalone_headers = ["Version.ID", "User.ID"] + [
    column
    for column in PREVIOUS_RUN_COLUMNS + NEW_POOLED_ALIAS_COLUMNS
    if column not in {"Version.ID", "User.ID"}
    and column not in MAIN_DATA_COLUMNS
]
# Preserve order while removing accidental repeats.
standalone_headers = list(dict.fromkeys(standalone_headers))
standalone_rows = [
    {column: row.get(column) for column in standalone_headers}
    for row in participant_features
]

duplicate_non_ids = (
    set(standalone_headers) & MAIN_DATA_COLUMNS
) - {"Version.ID", "User.ID"}
assert duplicate_non_ids == set()

# Across the merged main + standalone schema, every final_* suffix has an exact
# first_* counterpart available for pooled create_av + human_av analysis.
combined_columns = MAIN_DATA_COLUMNS | set(standalone_headers)
combined_first_suffixes = {
    column.removeprefix("first_")
    for column in combined_columns
    if column.startswith("first_")
}
combined_final_suffixes = {
    column.removeprefix("final_")
    for column in combined_columns
    if column.startswith("final_")
}
assert combined_first_suffixes == combined_final_suffixes

write_csv(OUTPUT_PATH, standalone_rows, standalone_headers)

print(f"Wrote: {OUTPUT_PATH.name}")
print(f"Rows: {len(standalone_rows)}")
print(f"Standalone columns: {len(standalone_headers)}")
print("Combined pooled first/final suffixes:", len(combined_first_suffixes))
print("Previous-run columns missing from combined schema:", missing_previous_combined)
print("Duplicated non-ID columns with main data:", sorted(duplicate_non_ids))


In [ ]:
# Final verification summary
rows = read_csv(OUTPUT_PATH)
headers = list(rows[0].keys())
MAIN_DATA_COLUMNS = set(['Version.ID', 'User.ID', 'date', 'time1', 'Fitbit1', 'time2', 'Fitbit2', 'X1.1', 'X1.2', 'X1.3', 'X1.4', 'X1.5', 'X1.6', 'X1.7', 'X2.1', 'X2.2', 'X2.3', 'X2.4', 'X2.5', 'X2.6', 'X2.7', 'x_1_1_1_the_robot_avatar_felt_tailored_to_me', 'x1_1_2_the_robot_avatar_felt_personalized', 'x1_1_3_customized_robot_avatar_to_me', 'x1_2_1_robot_avatar_expresses_my_unique_personality', 'x_1_2_2_it_makes_me_feel_unique', 'x1_2_3_the_robot_avatar_reflects_my_uniqueness', 'x1_3_1_i_like_the_robot_avatar', 'x1_3_2_my_idea_of_a_perfect_robot_avatar', 'x1_3_3_the_robot_avatar_looks_really_great', 'x1_4_1_he_design_of_the_robot_avatar_is_appealing', 'x1_4_2_the_design_of_the_robot_avatar_is_attractive', 'x1_4_3_the_design_of_the_robot_avatar_looks_great', 'x1_5_1_i_feel_a_strong_sense_of_closeness_to_this_robot_avatar', 'x1_5_2_i_feel_a_high_degree_of_personal_ownership_over_this_robot_avatar', 'x1_5_3_this_robot_avatar_incorporates_a_part_of_myself', 'x1_6_1_you_co_created_the_robot_avatar', 'X2.2.1', 'X2.2.2', 'X2.2.3', 'X2.3.1', 'X2.3.2', 'X2.3.3', 'X2.4.1', 'X2.4.2', 'X2.4.3', 'X2.5.1', 'X2.5.2', 'X2.5.3', 'X2.6.1', 'X2.6.2', 'X2.6.3', 'X2.7.1', 'X2.7.2', 'X2.7.3', 'X2.8.1', 'X2.8.2', 'X2.8.3', 'X2.8.4', 'X2.8.5', 'X2.9', 'X2.9.1', 'X2.9.2', 'X2.10', 'tip_robot', 'tip_human', 'tip_discount', 'X2.11.a', 'X2.11.b', 'X2.11.c', 'X2.12.1', 'X2.12.2', 'X2.12.3', 'X2.13.1', 'X2.13.2', 'X2.13.3', 'X2.14.1', 'X2.14.2', 'X2.14.3', 'X2.15', 'X2.16.1', 'X2.16.2', 'X2.16.3', 'X2.17', 'X2.18', 'X2.19', 'X2.20', 'X2.21', 'X2.22.a', 'X2.22.b', 'overall_experience', 'robot_interface', 'ordering_process', 'general_take_on_the_interaction_being_served_by_robots_and_humans', 'drawbacks_of_being_served_by_robots', 'what_elements_of_the_creation_process_you_liked_and_disliked', 'multiple_versions_of_avatars', 'why_did_you_choose_a_specific_one', 'did_you_find_the_creation_options_like_gender_logo_helpful_or_was_something_missing_would_you_change_anything_about_the_options', 'when_you_created_the_avatar_did_it_make_you_feel_that_like_i_did_this_like_this_is_my_own_creation', 'wd_flag_raw', 'x', 'x_2', 'x_3', 'x_4', 'x_5', 'x_6', 'x_7', 'x_8', 'x_9', 'x_10', 'x_11', 'x_12', 'x_13', 'x_14', 'x_15', 'x_16', 'x_17', 'x_18', 'x_19', 'x_20', 'x_21', 'x_22', 'x_23', 'x_24', 'x_25', 'x_26', 'x_27', 'x_28', 'x_29', 'x_30', 'x_31', 'x_32', 'x_33', 'x_34', 'x_35', 'x_36', 'x_37', 'x_38', 'x_39', 'x_40', 'x_41', 'x_42', 'x_43', 'x_44', 'x_45', 'x_46', 'x_47', 'x_48', 'x_49', 'x_50', 'x_51', 'x_52', 'x_53', 'x_54', 'x_55', 'x_56', 'x_57', 'x_58', 'x_59', 'x_60', 'x_61', 'x_62', 'x_63', 'x_64', 'x_65', 'x_66', 'x_67', 'x_68', 'x_69', 'x_70', 'x_71', 'x_72', 'x_73', 'x_74', 'x_75', 'x_76', 'x_77', 'x_78', 'x_79', 'x_80', 'x_81', 'x_82', 'x_83', 'x_84', 'x_85', 'x_86', 'x_87', 'x_88', 'x_89', 'x_90', 'x_91', 'x_92', 'x_93', 'x_94', 'x_95', 'x_96', 'x_97', 'x_98', 'x_99', 'x_100', 'x_101', 'x_102', 'x_103', 'x_104', 'x_105', 'x_106', 'x_107', 'x_108', 'x_109', 'x_110', 'x_111', 'x_112', 'x_113', 'x_114', 'x_115', 'x_116', 'x_117', 'x_118', 'x_119', 'x_120', 'x_121', 'x_122', 'x_123', 'x_124', 'x_125', 'x_126', 'x_127', 'x_128', 'x_129', 'x_130', 'x_131', 'x_132', 'x_133', 'x_134', 'x_135', 'x_136', 'x_137', 'x_138', 'x_139', 'x_140', 'x_141', 'x_142', 'x_143', 'x_144', 'x_145', 'x_146', 'x_147', 'x_148', 'x_149', 'x_150', 'x_151', 'x_152', 'x_153', 'x_154', 'x_155', 'x_156', 'wd_flag', 'version_code', 'version_enc', 'X2.16.3_r', 'flawless', 'sat', 'sat2', 'serv_qual', 'reuse', 'unique_perc', 'custom_perc', 'close_perc', 'like_perc', 'control_perc', 'effort_perc', 'design_attract', 'subj_stress1', 'racewhite', 'gender', 'revisit', 'age', 'educ', 'robot_familiar', 'robot_comfort', 'time_of_day', 'mapping_status', 'mapping_confidence', 'structure_match', 'match_diff_minutes', 'robot_session_id', 'session_file_name', 'avatar_1', 'avatar_2', 'avatar_3', 'avatar_1_gender', 'avatar_2_gender', 'avatar_3_gender', 'distinct_avatars_created_v1', 'interaction_intensity_v1', 'user_gender_label', 'first_avatar_gender_match', 'any_avatar_gender_match', 'final_avatar_gender_match', 'final_avatar_name', 'final_avatar_gender', 'final_avatar_emotion', 'final_avatar_color', 'final_avatar_attire', 'final_avatar_logo_binary', 'final_avatar_variant', 'avatar_female_binary', 'avatar_happy_binary', 'avatar_formal_binary', 'realism_proxy_0_100', 'realism_style', 'craftsmanship_complexity_proxy_0_100', 'craftsmanship_level', 'realism_proxy_z', 'craftsmanship_proxy_z', 'total_time_seconds', 'time_AVATAR_CUSTOMIZER_seconds', 'page_sequence', 'avatar_feature_available', 'analysis_note'])

condition_counts = {
    version: sum(row["Version.ID"] == version for row in rows)
    for version in ["v1r", "v2r", "v4r"]
}
human_first_available = sum(
    row["Version.ID"] == "v2r"
    and row.get("first_logo_binary") not in {"", None}
    for row in rows
)
human_final_available = sum(
    row["Version.ID"] == "v2r"
    and row.get("final_logo_binary") not in {"", None}
    for row in rows
)
duplicate_non_ids = (set(headers) & MAIN_DATA_COLUMNS) - {"Version.ID", "User.ID"}
combined_columns = set(headers) | MAIN_DATA_COLUMNS
first_suffixes = {c.removeprefix("first_") for c in combined_columns if c.startswith("first_")}
final_suffixes = {c.removeprefix("final_") for c in combined_columns if c.startswith("final_")}

print("Condition counts:", condition_counts)
print("human_av first-feature rows available:", human_first_available, "/ 64")
print("human_av final-feature rows available:", human_final_available, "/ 64")
print("Duplicate non-ID columns with main data:", sorted(duplicate_non_ids))
print("Combined matched first/final suffixes:", len(first_suffixes))

assert len(rows) == 190
assert condition_counts == {"v1r": 62, "v2r": 64, "v4r": 64}
assert human_first_available == 62
assert human_final_available == 0
assert duplicate_non_ids == set()
assert first_suffixes == final_suffixes
